# Project 6: Simple Neural Network Forward Propagation

**Dataset:** Iris flower classification dataset

**Goal:** Understand how a neural network turns raw input features into a
prediction by manually building the network and running **forward
propagation** on the Iris dataset with NumPy.

This notebook covers:
- **Task 1:** Load and prepare the Iris dataset for classification.
- **Task 2:** Build a simple neural network and perform forward propagation.
- **Task 3:** Apply an activation function and display the predicted output.


## Setup

Import the libraries we'll need. `numpy` does the math, `pandas` helps us look at the data, `matplotlib`/`seaborn` help us visualize it, and `scikit-learn` gives us an easy way to load the Iris dataset and split it into train/test sets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)  # for reproducible results
%matplotlib inline


## Task 1: Load and Prepare the Iris Dataset

We'll load the Iris dataset using `scikit-learn` (it ships with the same data
as the classic Iris CSV, including the one linked in the assignment:
https://gist.github.com/curran/a08a1080b88344b0c8a7).

The dataset has:
- **4 input features:** sepal length, sepal width, petal length, petal width (all in cm)
- **1 target:** the species of iris flower (Setosa, Versicolor, or Virginica)


In [ ]:
iris = load_iris()

X = iris.data                 # shape (150, 4) -> the 4 input features
y = iris.target                # shape (150,)  -> class labels: 0, 1, 2
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y, target_names)

print("Dataset shape:", X.shape)
print("Classes:", target_names)
df.head()


### (Optional) Loading from the CSV link instead

If you'd rather load the exact CSV from the Gist linked in the assignment,
you can uncomment and run the cell below instead of using `load_iris()`.
The code afterwards works the same either way, since it only cares about
`X` (features) and `y` (numeric class labels).


In [ ]:
# url = "https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/iris.csv"
# df_csv = pd.read_csv(url)
# X = df_csv.drop(columns=['species']).values
# y = df_csv['species'].astype('category').cat.codes.values
# target_names = df_csv['species'].astype('category').cat.categories.tolist()
# df_csv.head()


Let's quickly visualize the data to see how separable the three species are.

In [ ]:
sns.pairplot(df, hue='species', diag_kind='hist')
plt.suptitle("Iris Dataset - Feature Relationships", y=1.02)
plt.show()


### Prepare the data for the neural network

1. **Split** into training and test sets.
2. **Scale/standardize** the features so each one has mean 0 and standard
   deviation 1. Neural networks train and behave much better on standardized
   inputs.
3. **One-hot encode** the labels, since our output layer will have 3 neurons
   (one per class) and use softmax.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def one_hot(labels, n_classes=3):
    encoded = np.zeros((labels.shape[0], n_classes))
    encoded[np.arange(labels.shape[0]), labels] = 1
    return encoded

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

print("Training samples:", X_train_scaled.shape[0])
print("Test samples:", X_test_scaled.shape[0])
print("\nExample scaled input:", np.round(X_train_scaled[0], 3))
print("Example one-hot label:", y_train_oh[0], "->", target_names[y_train[0]])


## Task 2: Build a Simple Neural Network and Perform Forward Propagation

We'll build a small **feedforward neural network** with one hidden layer:

```
Input layer (4 neurons)  ->  Hidden layer (8 neurons)  ->  Output layer (3 neurons)
      features                    ReLU activation              Softmax activation
```

- **Input layer:** 4 neurons, one for each feature (sepal length/width, petal length/width).
- **Hidden layer:** 8 neurons. Each hidden neuron computes a weighted sum of
  all 4 inputs plus a bias, then applies an activation function.
- **Output layer:** 3 neurons, one per Iris species. Each output neuron
  produces a score ("logit") for its class, which we turn into probabilities
  with softmax.

### The math of one layer

For a layer with weight matrix $W$, bias vector $b$, and input $x$:

$$z = xW + b$$
$$a = \text{activation}(z)$$

We do this once to go from input -> hidden, and again to go from hidden -> output.


In [ ]:
n_inputs = X_train_scaled.shape[1]   # 4 features
n_hidden = 8                          # size of the hidden layer (our choice)
n_outputs = 3                         # 3 Iris classes

# Randomly initialize weights and biases.
# (In a real trained network these would be learned; here we use random
#  values just to demonstrate how forward propagation works.)
W1 = np.random.randn(n_inputs, n_hidden) * 0.5   # weights: input -> hidden
b1 = np.zeros((1, n_hidden))                     # bias for hidden layer

W2 = np.random.randn(n_hidden, n_outputs) * 0.5  # weights: hidden -> output
b2 = np.zeros((1, n_outputs))                    # bias for output layer

print("W1 shape (input->hidden):", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape (hidden->output):", W2.shape)
print("b2 shape:", b2.shape)


### Activation functions

We'll use **ReLU** for the hidden layer and **softmax** for the output layer.

In [ ]:
def relu(z):
    """ReLU activation: keeps positive values, zeroes out negative ones."""
    return np.maximum(0, z)

def softmax(z):
    """Softmax activation: turns raw scores (logits) into probabilities
    that sum to 1 across each row."""
    # subtract the max for numerical stability (avoids overflow)
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


### Forward propagation function

This function pushes an input batch `X` through the network: input -> hidden (ReLU) -> output (softmax).

In [ ]:
def forward_propagation(X, W1, b1, W2, b2):
    # --- Input layer -> Hidden layer ---
    Z1 = np.dot(X, W1) + b1     # weighted sum + bias
    A1 = relu(Z1)                # apply activation function

    # --- Hidden layer -> Output layer ---
    Z2 = np.dot(A1, W2) + b2    # weighted sum + bias
    A2 = softmax(Z2)             # apply activation function -> probabilities

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# Run forward propagation on the test set
predicted_probs, cache = forward_propagation(X_test_scaled, W1, b1, W2, b2)

print("Output shape:", predicted_probs.shape)  # (n_samples, 3 classes)
print("\nFirst 5 predicted probability rows (rows sum to 1):")
print(np.round(predicted_probs[:5], 3))


## Task 3: Apply the Activation Function and Display the Predicted Output

The softmax activation already gave us a probability for each of the 3
classes. Now we:

1. Pick the class with the **highest probability** as the prediction
   (`argmax`).
2. Compare predictions to the true labels.
3. Display the results in a readable table.

> **Note:** Because the weights `W1`/`W2` above are randomly initialized
> (not trained), accuracy will be roughly what you'd expect from random
> guessing (~33%) for 3 classes. This is expected — the point of this task
> is to understand *forward propagation*, not to train an accurate model.
> The optional bonus section at the end shows what happens once the network
> is actually trained.


In [ ]:
predicted_classes = np.argmax(predicted_probs, axis=1)

results_df = pd.DataFrame({
    "True Species": [target_names[i] for i in y_test],
    "Predicted Species": [target_names[i] for i in predicted_classes],
    "Prob. Setosa": np.round(predicted_probs[:, 0], 3),
    "Prob. Versicolor": np.round(predicted_probs[:, 1], 3),
    "Prob. Virginica": np.round(predicted_probs[:, 2], 3),
    "Correct?": y_test == predicted_classes
})

results_df.head(10)


In [ ]:
accuracy = np.mean(predicted_classes == y_test)
print(f"Accuracy with random (untrained) weights: {accuracy * 100:.2f}%")
print("This is close to chance level (~33%) since the network hasn't learned anything yet.")


### Visualize the predicted probabilities for one sample

In [ ]:
sample_idx = 0
sample_probs = predicted_probs[sample_idx]

plt.figure(figsize=(5, 4))
plt.bar(target_names, sample_probs, color=['#4C72B0', '#DD8452', '#55A868'])
plt.ylabel("Predicted probability")
plt.title(f"Forward-propagation output for test sample #{sample_idx}\n"
          f"(True label: {target_names[y_test[sample_idx]]})")
plt.ylim(0, 1)
plt.show()


## (Optional Bonus) Training the Network

Forward propagation alone won't produce accurate predictions because the
weights are random. To actually learn, the network needs **backpropagation**
and **gradient descent** to update `W1`, `b1`, `W2`, `b2` so that the
predicted probabilities get closer to the true labels.

Below is a minimal training loop using cross-entropy loss and manual
gradient descent, just to show how much forward propagation improves once
the weights are learned. This goes beyond the required tasks but helps tie
the concept together.


In [ ]:
def cross_entropy_loss(y_true_oh, y_pred):
    eps = 1e-9  # avoid log(0)
    return -np.mean(np.sum(y_true_oh * np.log(y_pred + eps), axis=1))

def backward_and_update(X, y_true_oh, cache, W2, learning_rate=0.1):
    m = X.shape[0]
    A1, A2, Z1 = cache["A1"], cache["A2"], cache["Z1"]

    # Gradient of loss w.r.t. output layer (softmax + cross-entropy simplifies nicely)
    dZ2 = (A2 - y_true_oh) / m
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * (Z1 > 0)  # derivative of ReLU
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2

# Re-initialize fresh weights for training
np.random.seed(42)
W1 = np.random.randn(n_inputs, n_hidden) * 0.5
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_outputs) * 0.5
b2 = np.zeros((1, n_outputs))

epochs = 500
learning_rate = 0.5
losses = []

for epoch in range(epochs):
    y_pred, cache = forward_propagation(X_train_scaled, W1, b1, W2, b2)
    loss = cross_entropy_loss(y_train_oh, y_pred)
    losses.append(loss)

    dW1, db1, dW2, db2 = backward_and_update(X_train_scaled, y_train_oh, cache, W2, learning_rate)
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 100 == 0:
        train_preds = np.argmax(y_pred, axis=1)
        train_acc = np.mean(train_preds == y_train)
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Train Accuracy: {train_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training Loss Over Time")
plt.show()


In [ ]:
# Forward propagation again, now with TRAINED weights
trained_probs, _ = forward_propagation(X_test_scaled, W1, b1, W2, b2)
trained_preds = np.argmax(trained_probs, axis=1)
trained_acc = np.mean(trained_preds == y_test)

print(f"Test accuracy with trained weights: {trained_acc * 100:.2f}%")

trained_results_df = pd.DataFrame({
    "True Species": [target_names[i] for i in y_test],
    "Predicted Species": [target_names[i] for i in trained_preds],
    "Correct?": y_test == trained_preds
})
trained_results_df.head(10)


## Summary

- **Task 1:** Loaded the Iris dataset, explored it, standardized the
  features, split into train/test sets, and one-hot encoded the labels.
- **Task 2:** Built a simple neural network (4 -> 8 -> 3 neurons) from
  scratch with NumPy and implemented forward propagation: `Z = XW + b`,
  `A = activation(Z)`, applied twice (input -> hidden, hidden -> output).
- **Task 3:** Applied **ReLU** in the hidden layer and **softmax** in the
  output layer, then used `argmax` to turn the predicted probabilities into
  a final class prediction, and displayed the results in a table.
- **Bonus:** Trained the network with backpropagation and gradient descent
  to show how forward propagation's output improves as the weights learn.
